# **Aprendizado _Online_**
<font size=3>

Até o momento, nossa jornada em Machine Learning baseou-se predominantemente no paradigma de *Batch Learning* (Aprendizado em Lote):
> Coletamos um conjunto de dados finito, treinamos um modelo estático visando minimizar uma função de perda global e, por fim, o implantamos. No entanto, o mundo real raramente é estático. Dados financeiros, monitoramento de sensores IoT, cliques de usuários e tráfego de rede geram **fluxos contínuos e potencialmente infinitos** de informação.

O **Aprendizado Online** (*Online Learning*) surge como a solução matemática e computacional para processar esses dados sequencialmente. Diferente da abordagem tradicional, aqui o modelo não tem acesso ao passado completo; **ele deve atualizar seus parâmetros internos instantaneamente** a cada nova observação, descartando o dado bruto logo em seguida.


## **1. O cenário de Aprendizado em Fluxo:**
<font size=3>
    
No aprendizado supervisionado tradicional (**_batch_**), assumimos que os dados são independentes e identicamente distribuídos (*i.i.d*) e que temos acesso a todo o conjunto $\mathcal{D} = \{(x_1, y_1), ..., (x_N, y_N)\}$ antes do treino começar.

<font size=3>

No **Aprendizado Online**, o cenário muda para um jogo sequencial entre o **Learner** (nosso modelo) e o **Ambiente** (a fonte de dados). O processo ocorre em rodadas discretas $t = 1, 2, ..., T$ (onde $T$ pode tender ao infinito):

1.  O Ambiente apresenta uma nova instância $x_t \in \mathbb{R}^d$ (vetor de características).
2.  O Modelo faz uma previsão $\hat{y}_t = f_{w_t}(x_t)$ usando seus parâmetros atuais $w_t$.
3.  O Ambiente revela o verdadeiro rótulo $y_t$.
4.  O Modelo sofre uma **perda instantânea** baseada em seu erro: $\mathcal L(\hat{y}_t,\, y_t)$.
5.  O Modelo atualiza seus parâmetros para $w_{t+1}$ visando melhorar nas rodadas futuras.

> **Nota Importante:** Diferente do batch, aqui **o tempo importa**. O modelo no tempo $t$ ($w_t$) só tem conhecimento dos dados observados até $t-1$.

### **1.1 A diferença matemática nos objetivos:**
<font size=3>

Podemos visualizar a mudança de paradigma comparando as estratégias matemáticas que guiam o aprendizado:
    
  - **No *batch learning* (Visão Global / Estática):**  
    Como temos acesso a todo o histórico de dados simultaneamente, podemos calcular e minimizar o erro médio global. Buscamos o vetor de pesos $w^*$ ideal que resolve todo o dataset de uma vez:

$$
    w^* = \operatorname*{argmin}_{w} \underbrace{\frac{1}{N} \sum_{i=1}^{N} \mathcal L(\hat y_i,\, y_i)}_{\text{Erro médio em todos os dados}} + \underbrace{\lambda\, R(w)}_{\text{Controle de complexidade}}
$$

  - **No *online learning* (Visão Local / Dinâmica):**  
    Sem acesso ao futuro e com memória limitada, não é possível minimizar uma soma global. A cada passo $t$, somos forçados a atualizar o modelo baseando-nos apenas no erro da instância atual (gradiente local), tentando melhorar para a próxima rodada:

$$
     w_{t+1} =  w_t - \eta \, \nabla \mathcal L(\hat y_t,\, y_t)
$$

Neste cenário, o conceito de "sucesso" muda. O objetivo teórico é minimizar o **Arrependimento (Regret)**. Pense no *Regret* como uma análise feita "pelo retrovisor": ao final do fluxo de dados, comparamos o erro total que nosso modelo acumulou (enquanto tentava aprender) com o erro que teríamos obtido se tivéssemos utilizado o **melhor modelo estático possível** desde o primeiro dia. Em suma, o *Regret* mede o "preço" que pagamos por termos que aprender durante o processo, em vez de começarmos já com a configuração ideal.


### **1.2 Restrições dos modelos _online_:**
<font size=3>
    
Para que um algoritmo seja considerado verdadeiramente "_online_", ele deve respeitar três restrições severas:
1. **Requisito de memória fixa:** O algoritmo não pode armazenar o histórico completo. A memória utilizada não deve crescer com o tempo $t$.

2. **Requisito de tempo constante:** O tempo para processar o dado $x_t$ e atualizar o modelo ($w_t \rightarrow w_{t+1}$) deve ser constante e rápido o suficiente para liberar o sistema antes que $x_{t+1}$ chegue.

3. **Previsão única (_one-pass_):** Na maioria dos casos, uma vez que o dado $x_t$ é processado, ele é descartado. Não podemos "voltar atrás" para re-treinar com ele.


## **2. A mudança de paradigma na avaliação:**
<font size=3>
    
No aprendizado de máquina clássico, a validação é estática: dividimos os dados em treino e teste (**Hold-out**) ou usamos **Validação Cruzada** (Cross-Validation). No **aprendizado _online_**, essas técnicas são inviáveis ou incorretas por dois motivos:

<font size=3>

1. **Dependência Temporal:** A ordem dos dados importa. Misturar (embaralhar) os dados para criar *folds* de validação destrói a estrutura temporal e esconde **_drifts_** (mudanças de padrão).
   
2. **Dados Infinitos:** Não existe um "conjunto de teste" fixo, pois novos dados chegam a todo momento.

### **2.1 Avaliação "_Prequencial_":**
<font size=3>
    
A metodologia padrão para avaliar modelos em fluxo é a **avaliação "_prequencial_"** (*predictive sequential*). Ela aproveita ao máximo cada instância de dado, alternando entre teste e treino. Para cada instante $t$ e novo dado $(x_t, y_t)$:
1. **Teste/predição:** Ocultamos o rótulo $y_t$ e pedimos para o modelo prever $\hat{y}_t$ usando seu conhecimento atual.

2. **Avaliação:** Comparamos $\hat{y}_t$ com o verdadeiro $y_t$ e registramos o erro.

3. **Treino:** Só agora revelamos $y_t$ ao modelo para que ele atualize seus pesos ($w_t \rightarrow w_{t+1}$).

Isso garante que o modelo seja sempre testado em dados que ele **nunca viu antes**, simulando perfeitamente o ambiente de produção.

### **2.2 Métricas dinâmicas:**
<font size=3>
    
Como o modelo online evolui com o tempo, **calcular a performance média desde o início do fluxo (média acumulada) pode ser enganoso**. Se o modelo teve um desempenho ruim no passado mas aprendeu e melhorou, ou se o padrão dos dados mudou drasticamente (*drift*), a média acumulada mascarará a realidade atual.

Por isso, utilizamos métricas baseadas em **janela deslizante** (*sliding window*). Calculamos a performance apenas sobre as últimas $W$ instâncias. A formulação matemática depende da natureza do problema:

- **Para classificação (acurácia móvel):**
medimos a taxa de acerto recente, dada por
$$
    \text{Acurácia}_t = \frac{1}{W} \sum_{i=t-W+1}^{t} \mathbb{I}(\hat{y}_i = y_i) \, ,
$$
onde $\mathbb{I}$ é a função indicadora (vale 1 se acertou, 0 se errou).

- **Para Regressão (erro médio móvel):**
medimos a distância média entre a previsão e o valor real,
$$
    \text{MSE}_t = \frac{1}{W} \sum_{i=t-W+1}^{t} (y_i - \hat{y}_i)^2 \, .
$$

>Essa abordagem nos permite plotar curvas de aprendizado que flutuam em tempo real, respondendo à pergunta crítica: *"Qual a confiabilidade do meu modelo **neste exato momento**?"*

## **3. Simulação de um fluxo de dados (_data stream_)**:
<font size=3>
    
Para testar nossos algoritmos, não podemos usar um *dataset* estático clássico. Precisamos de um **Gerador** (uma função Python que usa `yield`), capaz de produzir dados infinitamente ou sob demanda.

<font size=3>

Aqui, simularemos um problema de **classificação binária**, com dois atributos.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification

In [ ]:
class StreamData:

    def __init__(self, n_samples=2000, seed=42, module="sklearn"):

        self.n_samples = n_samples
        self.module = module
        self.current_step = 0

        self.X, self.y = make_classification(n_samples=n_samples,
                                             n_features=2,
                                             n_redundant=0,
                                             n_informative=2,
                                             n_clusters_per_class=1,
                                             flip_y=0.1,
                                             random_state=seed)

    def __iter__(self):
        self.current_step = 0
        return self

    def __next__(self):
        if self.current_step >= self.n_samples:
            raise StopIteration

        x = self.X[self.current_step]
        y = self.y[self.current_step]

        if self.module == "sklearn":
            x = x.reshape(1, -1)
            y = np.array([y])

        elif self.module == "river":
            x = {i:v for i, v in enumerate(x)}

        else:
            raise ValueError(f"Módulo '{self.module}' inválido! "
                             f"Opções suportadas: ['sklearn', 'river']")

        self.current_step += 1

        return x, y


In [ ]:
stream = StreamData(n_samples=5)

for x, y in stream:
    print(x, "\t", y)


## **4. Modelos de aprendizado _online_:**
<font size=3>

Nesta seção, exploraremos cinco abordagens distintas para resolver problemas em fluxo de dados. Cada modelo representa uma "família" diferente de algoritmos de Aprendizado de Máquina, adaptada para o cenário _online_.

### **4.1 Modelos Lineares (_SGD_):**
<font size=3>
    
O **Gradiente Descendente Estocástico** (*stochastic gradient discent* — SGD) não é um modelo em si, mas um algoritmo de otimização. Quando aplicado a modelos lineares (como SVM ou Regressão Logística), ele se torna a abordagem mais fundamental do aprendizado online.

Ele é popular por ser extremamente rápido e eficiente em memória, sendo a base de *redes neurais profundas*.

#### **4.1.1 Formulação matemática:**
<font size=3>
    
Em um modelo linear, a previsão $\hat{y}$ para uma instância $x$ é dada pelo produto escalar entre o vetor de características e o vetor de pesos $w$, somado a um viés $b$:
$$
    f(x) = w^T x + b
$$

No aprendizado online, nosso objetivo é atualizar $w$ a cada nova instância $(x_t,\, y_t)$ para minimizar uma função de perda $\mathcal L(\hat{y},\, y)$. A regra de atualização é:
$$
    w_{t+1} = w_t - \eta_t\,\nabla \mathcal L(w_t;\, x_t,\, y_t)
$$


#### **4.1.2 O algoritmo:**
<font size=3>

1. **Inicialização:** Começa com pesos aleatórios ou zerados.

2. **Passo iterativo:**
    * Recebe um novo dado $x_t$.
    * Faz a previsão usando os pesos atuais $w_t$;
    * Recebe o gabarito $y_t$ e calcula o erro.
    * Calcula o gradiente (a direção que reduziria esse erro);
    * Atualiza os pesos $w$ movendo-os levemente na direção oposta ao erro;
    * Descarta o dado $x_t$.
      

#### **4.1.3 Exemplo numérico:**
<font size=3>
    
Utilizaremos o [`SGDClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html) do Scikit-Learn. O segredo para o funcionamento online é o método `.partial_fit()`, que permite treinar o modelo em lotes incrementais (ou de tamanho 1) sem apagar o conhecimento anterior.

>**Observação:** os modelos desta aula que apresentam o *sufixo* $\mathtt{Classifier}$, também apresentam sua versão $\mathtt{Regressor}$, para realizarem tarefas de regressão.

In [ ]:
from sklearn.linear_model import SGDClassifier

In [ ]:
# simulando os dados:
stream = StreamData(n_samples=2000, seed=1)

# definindo o modelo:
model = SGDClassifier(loss='log_loss', learning_rate='constant', eta0=0.01, random_state=42)

window = 100
scores, moving_score = [], []

print("Iniciando treinamento incremental...")

for i, (x, y) in enumerate(stream):

    # predição:
    try: y_pred = model.predict(x)[0]
    except: y_pred = 0 # fallback para a primeira iteração (modelo não treinado)

    # avaliação:
    scores.append(1 if y_pred == y else 0)

    # cálculando a média móvel:
    if len(scores) > window: acc = np.mean(scores[-window:])
    else: acc = np.mean(scores)

    moving_score.append(acc)

    model.partial_fit(x, y, classes=[0, 1])

print(f"Média da acurácia das últimas {window} iterações: {moving_score[-1]:.2f}")

# visualização:
plt.figure(figsize=(12, 5))
plt.plot(moving_score, c='tab:blue')
plt.xlabel('Instâncias Processadas')
plt.ylabel(f'Acurácia (window={window})')
plt.grid(alpha=0.3)
plt.show()

### **4.2 Modelos Probabilísticos (_Gaussian Naïve Bayes_):**
<font size=3>
    
O Naive Bayes é um **classificador** baseado no Teorema de Bayes com a suposição "ingênua" de que todas as características ($x_0,\, x_1,\, \dots$) são independentes entre si dado a classe. Ele é naturalmente incremental.

#### **4.2.1 Formulação matemática:**
<font size=3>
    
O objetivo é encontrar a classe $y$ que maximiza a probabilidade a posteriori $P(y\mid x)$. Pelo Teorema de Bayes:
$$
    P(y \mid x_1, \dots, x_n) \propto P(y) \prod_{i=1}^{n} P(x_i \mid  y)
$$

Para dados contínuos, assumimos que $P(x_i \mid  y)$ segue uma **distribuição normal**:
$$
    P(x_i \mid y) = \frac{1}{\sqrt{2\pi\sigma_{y,i}^2}} \exp \left( -\frac{(x_i - \mu_{y,i})^2}{2\sigma_{y,i}^2} \right)
$$

No aprendizado online, não precisamos armazenar os dados. Precisamos apenas atualizar as **sstatísticas suficientes** a cada novo dado:
1. **Contagem de classes:** $N_y$ (quantas vezes vi a classe $y$);
2. **Soma dos valores:** $\sum x_i$ (para atualizar a média $\mu$);
3. **Soma dos quadrados:** $\sum x_i^2$ (para atualizar a variância $\sigma^2$).
   

#### **4.2.2 O algoritmo:**
<font size=3>
    
Diferente do SGD (que tenta minimizar um erro), o Naive Bayes apenas **conta**:
1. Recebe $(x_t,\, y_t)$;
2. Atualiza a média e a variância da classe $y_t$ usando as fórmulas de atualização incremental;
3. As médias antigas são descartadas; as novas representam o estado atual do conhecimento.


#### **4.2.3 Exemplo numérico:**
<font size=3>
    
Usaremos o [`GaussianNB`](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html) do Scikit-Learn. Note que o Naive Bayes geralmente aprende muito rápido no início, mas pode ter dificuldade em se adaptar a mudanças drásticas se não tiver um mecanismo de "esquecimento" (já que as contagens antigas têm muito peso).


In [ ]:
from sklearn.naive_bayes import GaussianNB

In [ ]:
# definindo o modelo:
model = GaussianNB(var_smoothing=0.01)

window = 100
scores, moving_score = [], []

print("Iniciando treinamento incremental...")

for i, (x, y) in enumerate(stream):

    # predição:
    try: y_pred = model.predict(x)[0]
    except: y_pred = 0 # fallback para a primeira iteração (modelo não treinado)

    # avaliação:
    scores.append(1 if y_pred == y else 0)

    # cálculando a média móvel:
    if len(scores) > window: acc = np.mean(scores[-window:])
    else: acc = np.mean(scores)

    moving_score.append(acc)

    model.partial_fit(x, y, classes=[0, 1])

print(f"Média da acurácia das últimas {window} iterações: {moving_score[-1]:.2f}")

# visualização:
plt.figure(figsize=(12, 5))
plt.plot(moving_score, c='tab:blue')
plt.xlabel('Instâncias Processadas')
plt.ylabel(f'Acurácia (window={window})')
plt.grid(alpha=0.3)
plt.show()

###  **4.3 Aprendizado baseado em instância (k-NN _online_):**
<font size=3>
    
O **k-Nearest Neighbors (k-NN)** é um algoritmo de aprendizado preguiçoso (*lazy learning*): ele não constrói um modelo matemático explícito (como pesos ou árvores) durante o treino. Em vez disso, ele apenas **memoriza** os dados.

No contexto *batch*, o k-NN memoriza tudo. No contexto *online*, isso é impossível (memória estouraria). A solução é usar uma **Janela Deslizante (Sliding Window)**: mantemos na memória apenas as últimas $W$ instâncias.

#### **4.3.1 Formulação matemática:**
<font size=3>
    
Dada uma nova instância $x_q$ (_query_), calculamos a distância entre ela e todas as instâncias $x_i$ presentes na janela atual $S_W$. A métrica mais comum é a **distância euclidiana**:
$$
    d(x_q,\, x_i) = \sqrt{\sum_{j=1}^{d} (x_{q,j} - x_{i,j})^2}
$$

Selecionamos os $k$ vizinhos com as menores distâncias. A previsão $\hat{y}_q$ é dada pela votação majoritária (moda) dos rótulos desses vizinhos:

$$\hat{y}_q = \operatorname*{argmax}_{c \in \text{Classes}} \sum_{i \in \text{Vizinhos}} \mathbb{I}(y_i = c)$$

#### **4.3.2 O algoritmo:**
<font size=3>
    
1. **Gestão de Janela:** O modelo possui um _buffer_ FIFO (*First-In, First-Out*) de tamanho máximo `window_size`.

2. **Treino (`.learn_one`):**
* Adiciona o novo par $(x_t,\, y_t)$ na janela.
* Se a janela estiver cheia, descarta a instância mais antiga $(x_{t-W},\, y_{t-W})$.
    <br>

3. **Previsão (`.predict_one`):**
* Varre a janela atual calculando distâncias.
* Encontra os $k$ vizinhos mais próximos.
* Retorna a classe majoritária.

> **O esquecimento do modelo:** Como o k-NN descarta fisicamente os dados antigos para dar lugar aos novos, ele se adapta a mudanças abruptas (*drift*) muito rapidamente. Assim que os dados do "velho conceito" saem da janela, o modelo para de errar.

#### **4.3.3 Exemplo numérico:**
<font size=3>
    
Usaremos o [`KNNClassifier`](https://riverml.xyz/dev/api/neighbors/KNNClassifier/) da biblioteca [`river`](https://riverml.xyz/latest/), a qual utiliza dicionários para representar a variável de atributos $x$.


In [ ]:
#pip install river==0.21

In [ ]:
from river.neighbors import KNNClassifier, SWINN
from river import utils, metrics

In [ ]:
# simulando o dataset para o river:
stream = StreamData(n_samples=2000, seed=1, module="river")

# definindo o modelo:
swinn = SWINN(maxlen=100, seed=1) # Sliding Window Incremental Nearest Neighbors
#                                   algoritmo de incremental de janela

model = KNNClassifier(n_neighbors=5, engine=swinn)

# definindo a métrica:
metric = utils.Rolling(metrics.Accuracy(), window_size=window)

moving_score = []

print("Iniciando treinamento incremental...")

for x, y in stream:
    y_pred = model.predict_one(x)

    metric.update(y, y_pred)

    moving_score.append(metric.get())

    # fitando o modelo:
    model.learn_one(x, y)

print(f"Média da acurácia das últimas {window} iterações: {metric.get():.2f}")

# visualização:
plt.figure(figsize=(12, 5))
plt.plot(moving_score, c='tab:blue')
plt.xlabel('Instâncias Processadas')
plt.ylabel(f'Acurácia (window={window})')
plt.grid(alpha=0.3)
plt.show()

### **4.4 Árvores de decisão incremental (_Hoeffding Tree_):**
<font size=3>
    
As árvores de decisão tradicionais (como CART) são **gananciosas**: elas precisam ver *todos* os dados disponíveis para calcular a impureza (Gini ou Entropia) e decidir qual o melhor atributo para dividir (*split*) um nó. Isso é impossível em um *stream* infinito.

A **_Hoeffding Tree_** contorna isso usando estatística. Ela mantém estatísticas nas folhas e pergunta: *"Quantos exemplos eu preciso ver para ter certeza matemática de que o atributo A é melhor que o atributo B?"*

#### **4.4.1 Formulação matemática:**
<font size=3>

O algoritmo baseia-se na **Desigualdade de Hoeffding**, que nos dá um limite probabilístico sobre o quanto a média observada de uma variável pode desviar de sua média real.

Suponha que queremos dividir um nó. Calculamos o Ganho de Informação ($G$) para todos os atributos.
Seja $G_a$ o ganho do melhor atributo e $G_b$ o ganho do segundo melhor.
A diferença é $\Delta G = G_a - G_b$.

O limite de Hoeffding ($\epsilon$) diz que, com probabilidade $1 - \delta$, a verdadeira média de uma variável aleatória de alcance $R$ não difere da média estimada após $n$ observações por mais de:
$$
    \epsilon = \sqrt{\frac{R^2 \ln(1/\delta)}{2n}}\, ,
$$
onde,
  * $\delta$: Probabilidade de erro aceitável (ex: 0.01 ou 1%).
  * $R$: Alcance da variável (para Entropia, $R = \log_2(\text{n\_classes})$).
  * $n$: Número de exemplos observados naquele nó até agora.
    
**A regra de decisão:**
Se $\Delta G > \epsilon$, então podemos afirmar com confiança $1 - \delta$ que o atributo $A$ é realmente superior ao $B$. Nesse momento, realizamos a divisão (split) e criamos novos nós.


#### **4.4.2 O algoritmo:**
<font size=3>

1.  **Inicialização:** A árvore começa apenas com a raiz (que é uma folha).

2.  **Fluxo:**
* Para cada instância $(x,\, y)$, ela "escorrega" pela árvore até chegar em uma folha.
* A folha atualiza seus contadores (estatísticas suficientes) com base em $x$ e $y$.
* A cada $N_{min}$ exemplos, a folha calcula os ganhos de informação dos atributos possíveis.
* Calcula o $\epsilon$ usando a fórmula acima.
* Se $(G_{melhor} - G_{segundo}) > \epsilon$, a folha se transforma em um nó de decisão e novas folhas vazias são criadas.

> **Vantagem:** O modelo cresce organicamente. Onde há mais dados e complexidade, a árvore se aprofunda. Onde os dados são simples, ela permanece rasa.

#### **4.4.3 Exemplo numérico:**
<font size=3>
    
Utilizaremos [`HoeffdingTreeClassifier`](https://riverml.xyz/dev/api/tree/HoeffdingTreeClassifier/) da biblioteca `river`.

In [ ]:
from river.tree import HoeffdingTreeClassifier

In [ ]:
# definindo o modelo:
model = HoeffdingTreeClassifier()

# definindo a métrica:
metric = utils.Rolling(metrics.Accuracy(), window_size=window)

moving_score = []

print("Iniciando treinamento incremental...")

for x, y in stream:
    y_pred = model.predict_one(x)

    metric.update(y, y_pred)

    moving_score.append(metric.get())

    # fitando o modelo:
    model.learn_one(x, y)

print(f"Média da acurácia das últimas {window} iterações: {metric.get():.2f}")

# visualização:
plt.figure(figsize=(12, 5))
plt.plot(moving_score, c='tab:blue')
plt.xlabel('Instâncias Processadas')
plt.ylabel(f'Acurácia (window={window})')
plt.grid(alpha=0.3)
plt.show()

### **4.5 _Ensembles online_ (_Adaptive Random Forest_):**
<font size=3>
    
*Random Forests* são extremamente populares no aprendizado *batch*. Elas funcionam treinando muitas árvores de decisão em subconjuntos aleatórios dos dados (**_bagging_**) e combinando suas previsões.

Mas como fazer "_bagging_" (_bootstrap aggregating_) se não temos o dataset fixo para sortear amostras? E como garantir que a floresta não fique obsoleta quando o conceito mudar? O **_adaptive random forest_ (ARF)** resolve isso com duas inovações: **OzaBagging** e **Árvores de Fundo**.


#### **4.5.1 Formulação matemática (OzaBagging):**
<font size=3>
    
No _bagging_ tradicional, sorteamos $N$ amostras com reposição de um *dataset* de tamanho $N$. A probabilidade de uma instância específica ser escolhida $k$ vezes segue uma **distribuição Binomial**.
Quando $N \to \infty$ (como em um *stream*), essa distribuição converge para uma **distribuição de Poisson** com $\lambda = 1$.

**O algoritmo online:**
Para cada nova instância $(x,\, y)$ e para cada árvore $T_m$ da floresta:
1. Sorteamos um valor $k \sim \text{Poisson}(1)$.
2. Treinamos a árvore $T_m$ com a instância $(x,\, y)$ exatas $k$ vezes.

Isso simula matematicamente o processo de *bootstrap* em tempo real.

#### **4.5.2 O mecanismo "_Adaptive_":**
<font size=3>
    
O ARF não é apenas uma floresta; é uma floresta que se "auto-poda". Cada árvore na floresta possui um detector de *drift* (geralmente o algoritmo ADWIN) acoplado a ela.

O processo funciona em 3 estados:
1.  **Estado estável:** A árvore treina e prevê normalmente.
2.  **Estado de aviso (_Warning_):** O detector percebe que o erro da árvore começou a subir. O algoritmo cria uma **"_Background Tree_"** (Árvore de Fundo) e começa a treiná-la do zero em paralelo.
3.  **Estado de _drift_:** O detector confirma que a árvore antiga não serve mais. O algoritmo **descarta a árvore antiga** e a substitui imediatamente pela **_Background Tree_** (que já estava treinando e está fresca).

> **Resultado:** A floresta nunca para de prever. Ela substitui seus galhos "velhos" por "jovens" dinamicamente, sem intervenção humana.

#### **4.5.3 Exemplo numérico:**
<font size=3>
    
Usaremos a classe [`ARFClassifier`](https://riverml.xyz/0.17.0/api/forest/ARFClassifier/) da biblioteca `river`.

  * `n_models`: Número de árvores na floresta (mais árvores = mais estável, mas mais lento).


In [ ]:
from river.forest import ARFClassifier

In [ ]:
# definindo o modelo:
model = ARFClassifier(n_models=10)

# definindo a métrica:
metric = utils.Rolling(metrics.Accuracy(), window_size=window)

moving_score = []

print("Iniciando treinamento incremental...")

for x, y in stream:
    y_pred = model.predict_one(x)

    metric.update(y, y_pred)

    moving_score.append(metric.get())

    # fitando o modelo:
    model.learn_one(x, y)

print(f"Média da acurácia das últimas {window} iterações: {metric.get():.2f}")

# visualização:
plt.figure(figsize=(12, 5))
plt.plot(moving_score, c='tab:blue')
plt.xlabel('Instâncias Processadas')
plt.ylabel(f'Acurácia (window={window})')
plt.grid(alpha=0.3)
plt.show()

## **5. Comparativo dos Modelos:**
<font size=3>
    
A tabela abaixo sintetiza as principais características, vantagens e desvantagens de cada abordagem estudada nesta aula.

<font size=3>

| Modelo | Família / Abordagem | Mecanismo Principal | Pontos Fortes (Pros) | Pontos Fracos (Cons) | Cenário Ideal |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **SGDClassifier** | Linear (Otimização) | Atualização de pesos via Gradiente Descendente. | • Extremamente rápido.<br>• Baixo consumo de memória.<br>• Bom para alta dimensionalidade. | • Só resolve problemas lineares.<br>• Sensível à escala dos dados e à *Learning Rate*. | Problemas com muitas features (ex: texto) e necessidade de velocidade extrema. |
| **Naïve Bayes** | Probabilístico | Atualização incremental de contagens (média/variância). | • Simples e rápido.<br>• Poucos hiperparâmetros.<br>• Convergência rápida. | • Assume independência entre features (raro na vida real).<br>• Pode sofrer com variância zero (requer *smoothing*). | Baseline inicial e classificação de texto/documentos. |
| **Hoeffding Tree** | Árvore de Decisão | Divisão de nós baseada em garantia estatística ($\epsilon$). | • Interpretabilidade (regras).<br>• Lida com não-linearidade.<br>• Robusto a ruído. | • Conservadora (lenta para reagir a *drifts*).<br>• Versão padrão não remove ramos antigos. | Dados estruturados complexos onde a explicabilidade é importante. |
| **KNN Online** | Lazy (Instância) | Janela Deslizante (FIFO) de memória curta. | • Adaptação **muito rápida** a *drifts* abruptos.<br>• Captura fronteiras de decisão complexas. | • **Lento na predição** (custo computacional alto).<br>• Sensível a *outliers* e à "Maldição da Dimensionalidade". | Mudanças de conceito frequentes e datasets com poucas dimensões. |
| **Adaptive Random Forest** | Ensemble (Floresta) | Bagging Online + Detectores de Drift (ADWIN). | • **Estado da arte** em acurácia.<br>• Robustez e estabilidade.<br>• Auto-gestão de *drift* (reset de árvores). | • Modelo "Caixa Preta".<br>• Custo computacional mais elevado (treina múltiplos modelos). | Sistemas críticos de produção onde a acurácia é a prioridade máxima. |

## **Referências:**
<font size=3>
    
 - **(15.1-15.4):** K. Faceli, Inteligência artificial: uma abordagem de aprendizado de máquina. Grupo Gen - LTC, 2011.
   